# Graph Inspection Notebook

This notebook loads and inspects the knowledge graph using the utility functions from `cli/commands/evals/utils.py`.

In [1]:
from pathlib import Path
import sys
import polars as pl
import random

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from cli.commands.evals.utils import (
    load_graph,
    load_node_metadata,
    compute_pagerank,
    pagerank_to_dataframe,
)

19:59:50 - LiteLLM:WARNING: get_model_cost_map.py:271 - LiteLLM: Failed to fetch remote model cost map from : Request URL is missing an 'http://' or 'https://' protocol.. Falling back to local backup.


[03/24/26 19:59:50] INFO     Using 'conf/logging.yml' as logging configuration. You can change this ]8;id=971744;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=83410;file:///Users/ayush/Documents/Zitnik_Lab/optimus/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py#270\270]8;;\
                             by setting the KEDRO_LOGGING_CONFIG environment variable accordingly.                 

## Load Graph

In [2]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
edges_path = Path("data/gold/kg/parquet/edges.parquet")

# load_graph now returns a MultiDiGraph; edge types are stored as edge attributes
G, node_types, edge_types = load_graph(nodes_path, edges_path)

                    INFO     Loaded 192035 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=807285;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=120980;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#47\47]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/24/26 20:01:48] INFO     Loaded 40983768 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=370259;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=363434;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#69\69]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     

## Basic Statistics

In [3]:
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"\nNode types ({len(node_types)}): {node_types}")
print(f"\nEdge types ({len(edge_types)}): {edge_types}")

Nodes: 192,035
Edges: 40,983,768

Node types (10): ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']

Edge types (26): ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Load Node Metadata

In [4]:
node_metadata = load_node_metadata(nodes_path)
node_metadata.head(10)

[03/24/26 20:02:00] INFO     Loaded metadata for 192035 nodes                                          ]8;id=253423;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=150040;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#106\106]8;;\

id,label,name
str,str,str
"""ENSG00000000003""","""GEN""","""tetraspanin 6"""
"""ENSG00000000005""","""GEN""","""tenomodulin"""
"""ENSG00000000419""","""GEN""","""dolichyl-phosphate mannosyltra…"
"""ENSG00000000457""","""GEN""","""SCY1 like pseudokinase 3"""
"""ENSG00000000460""","""GEN""","""FIGNL1 interacting regulator o…"
"""ENSG00000000938""","""GEN""","""FGR proto-oncogene, Src family…"
"""ENSG00000000971""","""GEN""","""complement factor H"""
"""ENSG00000001036""","""GEN""","""alpha-L-fucosidase 2"""
"""ENSG00000001084""","""GEN""","""glutamate-cysteine ligase cata…"


In [5]:
# Node counts by type
node_metadata.group_by("label").len().sort("len", descending=True)

label,len
str,u32
"""GEN""",61306
"""DIS""",36345
"""BPO""",25754
"""PHE""",19341
"""DRG""",16766
"""ANA""",14624
"""MFN""",10161
"""CCO""",4052
"""PWY""",2805


## Check Ghost Nodes

In [6]:
# Find nodes where id == GO_0009199 or GO_0009161
selected_ids = ["GO_0009199", "GO_0009161"]
selected_nodes = node_metadata.filter(pl.col("id").is_in(selected_ids))
selected_nodes

id,label,name
str,str,str


In [7]:
# Verify the IDs are absent from the raw nodes parquet
df = pl.read_parquet(nodes_path)
df.filter(pl.col("id").is_in(selected_ids))

id,label,properties
str,str,str


In [8]:
# Check raw edges parquet — must account for undirected edges, so look in
# both the `from` and `to` columns
edges_df = pl.read_parquet(edges_path)
edges_df.filter(
    pl.col("from").is_in(selected_ids)
    | (pl.col("to").is_in(selected_ids) & pl.col("undirected"))
)

from,to,label,relation,undirected,properties
str,str,str,str,bool,str


In [9]:
# Read in pagerank df and filter for selected nodes
pagerank_df = pl.read_csv("data/gold/evals/pagerank.csv")
pagerank_df.filter(pl.col("id").is_in(selected_ids))

rank,id,label,name,pagerank
i64,str,str,str,f64


## Debug Difference

In [10]:
print("Rows:", df.height)
print("Unique IDs:", df.select(pl.col("id").n_unique()).item())
print("Duplicate rows by ID:", df.height - df.select(pl.col("id").n_unique()).item())

Rows: 192035
Unique IDs: 192035
Duplicate rows by ID: 0


In [11]:
# Identify IDs that appear more than once
duplicate_ids = (
    df.group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .select("id")
)

# Join back to original data to get full metadata for those IDs
duplicated_nodes_table = (
    df.join(duplicate_ids, on="id", how="inner")
    .with_columns(
        pl.col("properties").str.json_path_match("$.name").alias("name")
    )
    .select(["id", "name", "label", "properties"])
    .sort("id")
)

# Save the table
output_file = Path("data/gold/evals/duplicated_nodes.csv")
if duplicated_nodes_table.height > 0:
    duplicated_nodes_table.write_csv(output_file)
    print(f"Found {duplicate_ids.height} unique duplicated IDs resulting in {duplicated_nodes_table.height} total rows.")
    print(f"Table saved to: {output_file}")

# Display the first few rows for inspection
duplicated_nodes_table.head(10)

id,name,label,properties
str,str,str,str


## Neighbor Consistency Check

In this section we compare, for a given set of node IDs, the neighbors reported
by the in-memory NetworkX `MultiDiGraph` `G` against neighbors obtained by directly
filtering the `edges.parquet` file. This is useful to debug potential discrepancies
between the graph representation and the underlying edge table.

**Key invariant for the Polars queries:** undirected edges are stored only once in
the parquet file (as a single `from → to` row with `undirected=true`). To find all
successors of node X from the raw table we therefore must match:
- rows where `from == X` (directed *or* undirected), **plus**
- rows where `to == X` and `undirected == true` (the implicit reverse arc).

Conversely, predecessors of X are:
- rows where `to == X` (directed *or* undirected), **plus**
- rows where `from == X` and `undirected == true`.

In [12]:
# Ensure edges dataframe is loaded
edges_df = pl.read_parquet(edges_path)


def get_out_neighbors_from_graph(G, node_id: str) -> set[str]:
    """Outgoing neighbors of `node_id` from a NetworkX MultiDiGraph.

    Returns nodes reached by edges node_id -> neighbor.
    If the node is not in the graph, returns an empty set.
    """
    if node_id not in G:
        return set()
    return set(G.successors(node_id))


def get_in_neighbors_from_graph(G, node_id: str) -> set[str]:
    """Incoming neighbors of `node_id` from a NetworkX MultiDiGraph.

    Returns nodes with edges neighbor -> node_id.
    If the node is not in the graph, returns an empty set.
    """
    if node_id not in G:
        return set()
    return set(G.predecessors(node_id))


def get_out_neighbors_from_edges_df(edges_df: pl.DataFrame, node_id: str) -> set[str]:
    """Outgoing neighbors of `node_id` from the raw edges table.

    Matches the MultiDiGraph semantics: undirected edges are stored once
    (from -> to, undirected=true) but imply arcs in both directions, so we
    collect:
      - `to`   where `from == node_id`  (forward arcs, directed or undirected)
      - `from` where `to == node_id` and `undirected == true`  (implicit reverse)
    """
    forward = (
        edges_df
        .filter(pl.col("from") == node_id)
        .select("to")
        .to_series()
        .to_list()
    )
    reverse = (
        edges_df
        .filter((pl.col("to") == node_id) & pl.col("undirected"))
        .select("from")
        .to_series()
        .to_list()
    )
    return set(forward) | set(reverse)


def get_in_neighbors_from_edges_df(edges_df: pl.DataFrame, node_id: str) -> set[str]:
    """Incoming neighbors of `node_id` from the raw edges table.

    Matches the MultiDiGraph semantics:
      - `from` where `to == node_id`  (direct predecessors, directed or undirected)
      - `to`   where `from == node_id` and `undirected == true`  (implicit reverse)
    """
    direct = (
        edges_df
        .filter(pl.col("to") == node_id)
        .select("from")
        .to_series()
        .to_list()
    )
    reverse = (
        edges_df
        .filter((pl.col("from") == node_id) & pl.col("undirected"))
        .select("to")
        .to_series()
        .to_list()
    )
    return set(direct) | set(reverse)


def compare_neighbors_for_nodes(
    G,
    edges_df: pl.DataFrame,
    node_ids: list[str],
) -> pl.DataFrame:
    """Compare incoming and outgoing neighbors from `G` vs. `edges_df`.

    For each node, returns:
    - outgoing counts from each source
    - outgoing intersection / mismatches
    - incoming counts from each source
    - incoming intersection / mismatches
    """
    records: list[dict] = []

    for node_id in node_ids:
        graph_out = get_out_neighbors_from_graph(G, node_id)
        graph_in = get_in_neighbors_from_graph(G, node_id)

        df_out = get_out_neighbors_from_edges_df(edges_df, node_id)
        df_in = get_in_neighbors_from_edges_df(edges_df, node_id)

        out_only_in_graph = sorted(graph_out - df_out)
        out_only_in_edges = sorted(df_out - graph_out)
        out_intersection = graph_out & df_out
        out_mismatch = graph_out != df_out

        in_only_in_graph = sorted(graph_in - df_in)
        in_only_in_edges = sorted(df_in - graph_in)
        in_intersection = graph_in & df_in
        in_mismatch = graph_in != df_in

        records.append(
            {
                "node_id": node_id,

                "graph_out_neighbors_count": len(graph_out),
                "edges_out_neighbors_count": len(df_out),
                "out_intersection_count": len(out_intersection),
                "out_mismatch": out_mismatch,
                "out_only_in_graph": " ".join(out_only_in_graph),
                "out_only_in_edges": " ".join(out_only_in_edges),

                "graph_in_neighbors_count": len(graph_in),
                "edges_in_neighbors_count": len(df_in),
                "in_intersection_count": len(in_intersection),
                "in_mismatch": in_mismatch,
                "in_only_in_graph": " ".join(in_only_in_graph),
                "in_only_in_edges": " ".join(in_only_in_edges),
            }
        )

    return pl.DataFrame(records)


# Sample 1000 random nodes and compare
random.seed(42)
random_ids = random.sample(list(G.nodes), 1000)
comparison_result = compare_neighbors_for_nodes(G, edges_df, random_ids)

# Save to CSV
comparison_result.write_csv("data/gold/evals/neighbor_comparison.csv")
comparison_result

node_id,graph_out_neighbors_count,edges_out_neighbors_count,out_intersection_count,out_mismatch,out_only_in_graph,out_only_in_edges,graph_in_neighbors_count,edges_in_neighbors_count,in_intersection_count,in_mismatch,in_only_in_graph,in_only_in_edges
str,i64,i64,i64,bool,str,str,i64,i64,i64,bool,str,str
"""GO_0098703""",32,32,32,false,"""""","""""",28,28,28,false,"""""",""""""
"""ENSG00000230120""",44,44,44,false,"""""","""""",44,44,44,false,"""""",""""""
"""ENSG00000132275""",533,533,533,false,"""""","""""",553,553,553,false,"""""",""""""
"""UBERON_0037480""",0,0,0,false,"""""","""""",1,1,1,false,"""""",""""""
"""UBERON_0003723""",0,0,0,false,"""""","""""",1,1,1,false,"""""",""""""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""UBERON_0013489""",0,0,0,false,"""""","""""",1,1,1,false,"""""",""""""
"""GO_1905281""",4,4,4,false,"""""","""""",2,2,2,false,"""""",""""""
"""ENSG00000207797""",52,52,52,false,"""""","""""",54,54,54,false,"""""",""""""


In [ ]:
G["GO_0098703"][""]

AdjacencyView({'ENSG00000006283': {0: {'label': 'BPO-PRO'}}, 'ENSG00000058668': {0: {'label': 'BPO-PRO'}}, 'ENSG00000074621': {0: {'label': 'BPO-PRO'}}, 'ENSG00000081248': {0: {'label': 'BPO-PRO'}}, 'ENSG00000100346': {0: {'label': 'BPO-PRO'}}, 'ENSG00000100678': {0: {'label': 'BPO-PRO'}}, 'ENSG00000102001': {0: {'label': 'BPO-PRO'}}, 'ENSG00000111199': {0: {'label': 'BPO-PRO'}}, 'ENSG00000118160': {0: {'label': 'BPO-PRO'}}, 'ENSG00000127412': {0: {'label': 'BPO-PRO'}}, 'ENSG00000130054': {0: {'label': 'BPO-PRO'}}, 'ENSG00000134160': {0: {'label': 'BPO-PRO'}}, 'ENSG00000140090': {0: {'label': 'BPO-PRO'}}, 'ENSG00000141837': {0: {'label': 'BPO-PRO'}}, 'ENSG00000142185': {0: {'label': 'BPO-PRO'}}, 'ENSG00000148408': {0: {'label': 'BPO-PRO'}}, 'ENSG00000151067': {0: {'label': 'BPO-PRO'}}, 'ENSG00000153956': {0: {'label': 'BPO-PRO'}}, 'ENSG00000155886': {0: {'label': 'BPO-PRO'}}, 'ENSG00000157388': {0: {'label': 'BPO-PRO'}}, 'ENSG00000165125': {0: {'label': 'BPO-PRO'}}, 'ENSG00000167723': 